# TabTransformer + LightGBM + XGBoost 앙상블

**전략**: TabTransformer로 범주형 변수의 컨텍스트 임베딩을 학습 → 임베딩을 GBDT 입력으로 사용  
**모델 구성**: TabTransformer (임베딩 추출) → LightGBM / XGBoost → Soft Voting 앙상블

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

Device: mps


## 1. 데이터 로딩 및 피처 준비

In [4]:
data_path = './data/kaggle_data/'
train = pd.read_csv(data_path + 'train.csv')
test  = pd.read_csv(data_path + 'test.csv')
submission = pd.read_csv(data_path + 'sample_submission.csv')

print(f'Train: {train.shape}, Test: {test.shape}')

# 피처 정의
cat_features  = ['Driver', 'Compound', 'Race']
cont_features = [
    'Year', 'LapNumber', 'Stint', 'TyreLife', 'Position',
    'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
    'RaceProgress', 'Position_Change', 'PitStop'
]
target = 'PitNextLap'

# 범주형 인코딩
cat_encoders = {}
cat_dims = []
emb_dim_map = {'Driver': 32, 'Compound': 4, 'Race': 8}
cat_emb_dims = []

all_data = pd.concat([train, test], ignore_index=True)

for col in cat_features:
    le = LabelEncoder()
    all_data[col + '_enc'] = le.fit_transform(all_data[col])
    cat_encoders[col] = le
    cat_dims.append(len(le.classes_))
    cat_emb_dims.append(emb_dim_map[col])

# 연속형 스케일링 (train 기준 fit)
scaler = StandardScaler()
train_idx = all_data.index[:len(train)]
test_idx  = all_data.index[len(train):]
scaler.fit(all_data.loc[train_idx, cont_features])
all_data[cont_features] = scaler.transform(all_data[cont_features])

train_df = all_data.iloc[:len(train)].copy()
test_df  = all_data.iloc[len(train):].copy()

X_train = train_df
y_train = train[target].values.astype(np.float32)

cat_enc_cols = [c + '_enc' for c in cat_features]
print(f'Cat 피처: {cat_enc_cols}')
print(f'Cont 피처 수: {len(cont_features)}')
print(f'Target 분포: {y_train.mean():.3f} (positive rate)')

Train: (439140, 16), Test: (188165, 15)
Cat 피처: ['Driver_enc', 'Compound_enc', 'Race_enc']
Cont 피처 수: 11
Target 분포: 0.199 (positive rate)


## 2. TabTransformer 모델 정의

In [5]:
class TabTransformer(nn.Module):
    """
    범주형 피처 → Embedding → Transformer Encoder → 연속형 피처와 concat → MLP 분류기
    임베딩 추출 모드(return_embeddings=True)로 GBDT의 입력 피처로 활용
    """
    def __init__(self, cat_dims, cat_emb_dims, n_continuous,
                 n_heads=4, n_layers=2, dim_feedforward=256, dropout=0.1):
        super().__init__()
        
        # 범주형 임베딩 레이어
        self.embeddings = nn.ModuleList([
            nn.Embedding(n_cats, emb_dim)
            for n_cats, emb_dim in zip(cat_dims, cat_emb_dims)
        ])
        
        # 임베딩 차원을 동일하게 맞추기 위한 projection (Transformer 입력 통일)
        self.emb_dim = max(cat_emb_dims)
        self.projections = nn.ModuleList([
            nn.Linear(d, self.emb_dim) if d != self.emb_dim else nn.Identity()
            for d in cat_emb_dims
        ])
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.emb_dim,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # 분류 헤드 (TabTransformer 단독 학습용)
        total_dim = self.emb_dim * len(cat_dims) + n_continuous
        self.classifier = nn.Sequential(
            nn.LayerNorm(total_dim),
            nn.Dropout(dropout),
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
    
    def get_embeddings(self, cat_x, cont_x):
        """임베딩 추출 (GBDT 입력용)"""
        embs = [proj(emb(cat_x[:, i]))
                for i, (emb, proj) in enumerate(zip(self.embeddings, self.projections))]
        emb_seq = torch.stack(embs, dim=1)          # (B, n_cat, emb_dim)
        emb_seq = self.transformer(emb_seq)          # (B, n_cat, emb_dim)
        emb_flat = emb_seq.flatten(1)                # (B, n_cat * emb_dim)
        return torch.cat([emb_flat, cont_x], dim=1) # (B, total_dim)
    
    def forward(self, cat_x, cont_x):
        features = self.get_embeddings(cat_x, cont_x)
        return self.classifier(features).squeeze(-1)


tab_model = TabTransformer(
    cat_dims=cat_dims,
    cat_emb_dims=cat_emb_dims,
    n_continuous=len(cont_features),
    n_heads=4, n_layers=2, dim_feedforward=256, dropout=0.1
).to(device)

n_params = sum(p.numel() for p in tab_model.parameters() if p.requires_grad)
print(f'TabTransformer 학습 파라미터: {n_params:,}')
print(f'임베딩 출력 차원: {tab_model.emb_dim * len(cat_dims) + len(cont_features)}')

TabTransformer 학습 파라미터: 85,275
임베딩 출력 차원: 107


## 3. TabTransformer 학습 (임베딩 사전 학습)

In [6]:
class TabDataset(Dataset):
    def __init__(self, df, cat_cols, cont_cols, labels=None):
        self.cat  = torch.LongTensor(df[cat_cols].values)
        self.cont = torch.FloatTensor(df[cont_cols].values)
        self.labels = torch.FloatTensor(labels) if labels is not None else None
    
    def __len__(self):
        return len(self.cat)
    
    def __getitem__(self, idx):
        if self.labels is not None:
            return self.cat[idx], self.cont[idx], self.labels[idx]
        return self.cat[idx], self.cont[idx]


# StratifiedKFold (5-fold, fold 0 = validation)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(skf.split(X_train, y_train))
tr_idx, val_idx = fold_splits[0]

tr_dataset  = TabDataset(X_train.iloc[tr_idx],  cat_enc_cols, cont_features, y_train[tr_idx])
val_dataset = TabDataset(X_train.iloc[val_idx], cat_enc_cols, cont_features, y_train[val_idx])
test_dataset = TabDataset(test_df, cat_enc_cols, cont_features)

tr_loader   = DataLoader(tr_dataset,   batch_size=2048, shuffle=True)
val_loader  = DataLoader(val_dataset,  batch_size=4096, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

print(f'Train: {len(tr_dataset):,}, Val: {len(val_dataset):,}, Test: {len(test_dataset):,}')

Train: 351,312, Val: 87,828, Test: 188,165


In [7]:
pos_rate   = y_train.mean()
pos_weight = torch.tensor([(1 - pos_rate) / pos_rate], dtype=torch.float32).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(tab_model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

n_epochs = 20
patience  = 5
best_auc  = 0
no_improve = 0
history = {'loss': [], 'auc': []}

for epoch in range(n_epochs):
    # 학습
    tab_model.train()
    total_loss = 0
    for cats, conts, labels in tr_loader:
        cats, conts, labels = cats.to(device), conts.to(device), labels.to(device)
        logits = tab_model(cats, conts)
        loss   = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(tr_loader)
    
    # 검증
    tab_model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for cats, conts, labels in val_loader:
            cats, conts = cats.to(device), conts.to(device)
            probs = torch.sigmoid(tab_model(cats, conts)).cpu().numpy()
            val_preds.extend(probs)
            val_labels.extend(labels.numpy())
    
    auc = roc_auc_score(val_labels, val_preds)
    history['loss'].append(avg_loss)
    history['auc'].append(auc)
    print(f'Epoch {epoch+1:02d}/{n_epochs} | Loss: {avg_loss:.4f} | Val AUC: {auc:.4f}')
    
    if auc > best_auc:
        best_auc = auc
        no_improve = 0
        torch.save(tab_model.state_dict(), 'best_tabtransformer.pth')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'\nEarly stopping. Best AUC: {best_auc:.4f}')
            break

print(f'\nTabTransformer Best Val AUC: {best_auc:.4f}')

Epoch 01/20 | Loss: 0.9149 | Val AUC: 0.8273
Epoch 02/20 | Loss: 0.8130 | Val AUC: 0.8644
Epoch 03/20 | Loss: 0.7543 | Val AUC: 0.8892
Epoch 04/20 | Loss: 0.7065 | Val AUC: 0.9041
Epoch 05/20 | Loss: 0.6744 | Val AUC: 0.9100
Epoch 06/20 | Loss: 0.6530 | Val AUC: 0.9142
Epoch 07/20 | Loss: 0.6399 | Val AUC: 0.9188
Epoch 08/20 | Loss: 0.6276 | Val AUC: 0.9205
Epoch 09/20 | Loss: 0.6188 | Val AUC: 0.9206
Epoch 10/20 | Loss: 0.6139 | Val AUC: 0.9229
Epoch 11/20 | Loss: 0.6078 | Val AUC: 0.9245
Epoch 12/20 | Loss: 0.6044 | Val AUC: 0.9236
Epoch 13/20 | Loss: 0.6024 | Val AUC: 0.9244
Epoch 14/20 | Loss: 0.5986 | Val AUC: 0.9255
Epoch 15/20 | Loss: 0.5977 | Val AUC: 0.9251
Epoch 16/20 | Loss: 0.5983 | Val AUC: 0.9255
Epoch 17/20 | Loss: 0.5964 | Val AUC: 0.9251
Epoch 18/20 | Loss: 0.5940 | Val AUC: 0.9253
Epoch 19/20 | Loss: 0.5966 | Val AUC: 0.9255
Epoch 20/20 | Loss: 0.5960 | Val AUC: 0.9254

TabTransformer Best Val AUC: 0.9255


## 4. TabTransformer 임베딩 추출

In [ ]:
def extract_embeddings(model, df, cat_cols, cont_cols, device, batch_size=4096):
    """학습된 TabTransformer로부터 피처 임베딩 추출"""
    model.eval()
    dataset = TabDataset(df, cat_cols, cont_cols)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_embs = []
    
    with torch.no_grad():
        for cats, conts in loader:
            cats, conts = cats.to(device), conts.to(device)
            embs = model.get_embeddings(cats, conts)
            all_embs.append(embs.cpu().numpy())
    
    return np.vstack(all_embs)


# Best 모델 로드
tab_model.load_state_dict(torch.load('best_tabtransformer.pth', weights_only=True))

# 전체 train/test 임베딩 추출
full_tr_dataset = TabDataset(X_train, cat_enc_cols, cont_features)
full_tr_loader  = DataLoader(full_tr_dataset, batch_size=4096, shuffle=False)

X_emb_train = extract_embeddings(tab_model, X_train, cat_enc_cols, cont_features, device)
X_emb_test  = extract_embeddings(tab_model, test_df, cat_enc_cols, cont_features, device)

print(f'Train 임베딩: {X_emb_train.shape}')
print(f'Test  임베딩: {X_emb_test.shape}')

Train 임베딩: (439140, 107)
Test  임베딩: (188165, 107)


: 

## 5. LightGBM 학습

In [ ]:
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'scale_pos_weight': (1 - pos_rate) / pos_rate,
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42
}

# Fold 0 기준 train/val 분할
X_lgb_tr  = X_emb_train[tr_idx]
X_lgb_val = X_emb_train[val_idx]
y_lgb_tr  = y_train[tr_idx]
y_lgb_val = y_train[val_idx]

lgb_train = lgb.Dataset(X_lgb_tr,  label=y_lgb_tr)
lgb_val   = lgb.Dataset(X_lgb_val, label=y_lgb_val, reference=lgb_train)

callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=callbacks
)

lgb_val_pred = lgb_model.predict(X_lgb_val)
lgb_auc = roc_auc_score(y_lgb_val, lgb_val_pred)
print(f'LightGBM Val AUC: {lgb_auc:.4f}')

## 6. XGBoost 학습

In [ ]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': (1 - pos_rate) / pos_rate,
    'tree_method': 'hist',
    'device': 'cpu',   # XGBoost MPS 미지원
    'n_jobs': -1,
    'random_state': 42,
    'verbosity': 0
}

dtrain = xgb.DMatrix(X_lgb_tr,  label=y_lgb_tr)
dval   = xgb.DMatrix(X_lgb_val, label=y_lgb_val)

xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100
)

xgb_val_pred = xgb_model.predict(dval)
xgb_auc = roc_auc_score(y_lgb_val, xgb_val_pred)
print(f'XGBoost Val AUC: {xgb_auc:.4f}')

## 7. 앙상블 및 성능 비교

In [ ]:
# TabTransformer 단독 val 예측 (fold 0)
tab_model.eval()
tab_val_preds = []
val_dataset_fold = TabDataset(X_train.iloc[val_idx], cat_enc_cols, cont_features)
val_loader_fold  = DataLoader(val_dataset_fold, batch_size=4096, shuffle=False)
with torch.no_grad():
    for cats, conts in val_loader_fold:
        cats, conts = cats.to(device), conts.to(device)
        probs = torch.sigmoid(tab_model(cats, conts)).cpu().numpy()
        tab_val_preds.extend(probs)
tab_val_preds = np.array(tab_val_preds)

tab_auc = roc_auc_score(y_lgb_val, tab_val_preds)

# 앙상블: Soft Voting (동일 가중치)
ensemble_pred = (lgb_val_pred + xgb_val_pred + tab_val_preds) / 3
ensemble_auc  = roc_auc_score(y_lgb_val, ensemble_pred)

# 결과 요약
print('=' * 45)
print(f'{"모델":<25} {"Val AUC":>10}')
print('=' * 45)
print(f'{"TabTransformer (단독)":<25} {tab_auc:>10.4f}')
print(f'{"LightGBM":<25} {lgb_auc:>10.4f}')
print(f'{"XGBoost":<25} {xgb_auc:>10.4f}')
print(f'{"앙상블 (Soft Voting)":<25} {ensemble_auc:>10.4f}')
print('=' * 45)

# F1 비교 (threshold=0.5)
for name, preds in [('TabTransformer', tab_val_preds), ('LightGBM', lgb_val_pred),
                     ('XGBoost', xgb_val_pred), ('Ensemble', ensemble_pred)]:
    f1 = f1_score(y_lgb_val, (preds > 0.5).astype(int))
    print(f'{name:<25} F1: {f1:.4f}')

## 8. 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# TabTransformer 학습 곡선
axes[0].plot(history['loss'], label='Loss')
ax2 = axes[0].twinx()
ax2.plot(history['auc'], color='orange', label='AUC')
axes[0].set_title('TabTransformer Training')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
ax2.set_ylabel('AUC')

# ROC Curve 비교
for name, preds in [('TabTransformer', tab_val_preds), ('LightGBM', lgb_val_pred),
                     ('XGBoost', xgb_val_pred), ('Ensemble', ensemble_pred)]:
    fpr, tpr, _ = roc_curve(y_lgb_val, preds)
    auc = roc_auc_score(y_lgb_val, preds)
    axes[1].plot(fpr, tpr, label=f'{name} ({auc:.4f})')
axes[1].plot([0,1],[0,1],'k--', alpha=0.3)
axes[1].set_title('ROC Curve 비교')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=9)

# 앙상블 Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_lgb_val, (ensemble_pred > 0.5).astype(int),
    display_labels=['No Pit', 'Pit'], ax=axes[2]
)
axes[2].set_title('Ensemble Confusion Matrix')

plt.tight_layout()
plt.show()

## 9. Test 추론 및 제출 파일 생성

In [ ]:
# TabTransformer test 예측
tab_test_preds = []
with torch.no_grad():
    for cats, conts in test_loader:
        cats, conts = cats.to(device), conts.to(device)
        probs = torch.sigmoid(tab_model(cats, conts)).cpu().numpy()
        tab_test_preds.extend(probs)
tab_test_preds = np.array(tab_test_preds)

# LightGBM test 예측
lgb_test_pred = lgb_model.predict(X_emb_test)

# XGBoost test 예측
dtest = xgb.DMatrix(X_emb_test)
xgb_test_pred = xgb_model.predict(dtest)

# 앙상블
ensemble_test = (lgb_test_pred + xgb_test_pred + tab_test_preds) / 3

# 제출 파일
submission['PitNextLap'] = ensemble_test
submission.to_csv('submission_tabtransformer_ensemble.csv', index=False)
print(f'제출 파일 저장: submission_tabtransformer_ensemble.csv')
print(f'예측 통계:\n{submission["PitNextLap"].describe()}')